# En masse, make FINAL GIFs from FINAL masks.

In [1]:
# # Original code as given by Maddie before 2026 02 12, not altered in any way.

# import os
# import numpy as np
# from PIL import Image
# import imageio.v2 as imageio  # pip install imageio

# # sessions_to_process = [
# #     '2025_10_14_10_25_19_trial_1_TC',
# #     '2025_10_14_14_13_15_trial_1_TC',
# #     '2025_10_15_10_09_28_trial_1_TC',
# #     '2025_10_15_14_16_21_trial_1_TC',
# #     '2025_10_16_09_59_15_trial_1_TC',
# #     '2025_10_16_14_12_17_trial_1_TC',
# #     '2025_10_17_10_05_03_trial_1_TC',
# #     '2025_10_17_14_18_23_trial_1_TC',
# #     '2025_10_17_14_45_34_trial_1_TP'
# # ]

# sessions_to_process = ['2025_10_17_14_45_34_trial_1_TP']

# MASK_DIR = "/n/holylabs/gershman_lab/Users/zkelso/Masks"
# FRAMES_ROOT = "/n/netscratch/gershman_lab/Lab/zkelso/temporary_jpgs/unlabeled-data"
# GIF_OUT_DIR = "/n/netscratch/gershman_lab/Lab/zkelso/FINAL_GIF"

# fps = 14
# mask_color = (0, 255, 0)  # red
# mask_alpha = 70           # transparency


# def find_final_mask_file(base_sess):
#     """
#     Find the mask file in MASK_DIR that:
#       - contains the base_sess string
#       - contains 'FINAL'
#       - ends with '.npz'
#     Returns full path, or None if not found.
#     """
#     candidates = [
#         f for f in os.listdir(MASK_DIR)
#         if base_sess in f and "FINAL" in f and f.endswith(".npz")
#     ]
#     if not candidates:
#         print(f"[{base_sess}] No FINAL mask .npz file found in {MASK_DIR}")
#         return None
#     if len(candidates) > 1:
#         print(f"[{base_sess}] Warning: multiple FINAL mask files found, using first:")
#         for c in candidates:
#             print("   ", c)
#     chosen = candidates[0]
#     return os.path.join(MASK_DIR, chosen)


# def process_one_session(base_sess):
#     npz_path = find_final_mask_file(base_sess)
#     if npz_path is None:
#         return

#     # Session+region name comes from the npz filename (without extension)
#     sess_full = os.path.splitext(os.path.basename(npz_path))[0]
#     sess_full = sess_full.split('_frame')[0]
#     print(f"[{base_sess}] Using mask file: {npz_path}")
#     print(f"[{base_sess}] Derived inner session folder: {sess_full}")

#     frames_dir = os.path.join(FRAMES_ROOT, base_sess, sess_full)
#     gif_path = os.path.join(GIF_OUT_DIR, f"{sess_full}_FINAL.gif")

#     if not os.path.isdir(frames_dir):
#         print(f"[{base_sess}] Frames directory does not exist: {frames_dir}")
#         return

#     print(f"[{base_sess}] Frames dir: {frames_dir}")
#     print(f"[{base_sess}] GIF output: {gif_path}")

#     # Load npz
#     data = np.load(npz_path)
#     masks = data["masks"]  # expected shape: (n_frames, H, W)
#     n_frames = masks.shape[0]

#     print(f"[{base_sess}] Masks shape: {masks.shape}")

#     gif_frames = []

#     for i in range(n_frames):
#         fname = f"{i:05d}.jpeg"
#         fpath = os.path.join(frames_dir, fname)
#         if not os.path.exists(fpath):
#             print(f"[{base_sess}] Warning: frame not found: {fpath}, skipping")
#             continue

#         frame = Image.open(fpath).convert("RGB")
#         frame_w, frame_h = frame.size

#         mask = masks[i]

#         # Convert mask to boolean
#         if mask.dtype != bool:
#             mask_bool = mask > 0
#         else:
#             mask_bool = mask

#         h_mask, w_mask = mask_bool.shape

#         # Resize mask to match frame size if needed
#         if (w_mask, h_mask) != (frame_w, frame_h):
#             mask_img = Image.fromarray(mask_bool.astype(np.uint8) * 255)
#             mask_img = mask_img.resize((frame_w, frame_h), resample=Image.NEAREST)
#             mask_bool = np.array(mask_img) > 0
#             h_mask, w_mask = mask_bool.shape

#         # Create overlay
#         overlay = Image.new("RGBA", (frame_w, frame_h), (0, 0, 0, 0))
#         overlay_pixels = overlay.load()

#         for y in range(h_mask):
#             for x in range(w_mask):
#                 if mask_bool[y, x]:
#                     overlay_pixels[x, y] = mask_color + (mask_alpha,)

#         frame_rgba = frame.convert("RGBA")
#         composite = Image.alpha_composite(frame_rgba, overlay)
#         composite_rgb = composite.convert("RGB")
#         gif_frames.append(np.array(composite_rgb))

#         if (i + 1) % 100 == 0:
#             print(f"[{base_sess}] Processed {i+1}/{n_frames} frames")

#     if gif_frames:
#         duration_per_frame = 1.0 / fps
#         os.makedirs(GIF_OUT_DIR, exist_ok=True)
#         imageio.mimsave(gif_path, gif_frames, duration=duration_per_frame)
#         print(f"[{base_sess}] Saved GIF to {gif_path}")
#     else:
#         print(f"[{base_sess}] No frames to save – check your paths and masks.")


# def main():
#     for sess in sessions_to_process:
#         print(f"=== Processing session: {sess} ===")
#         process_one_session(sess)


# if __name__ == "__main__":
#     main()

In [1]:
# Very slightly updated code to select most recently created mask file that fits pre-existing criteria (e.g. "FINAL" in the name)
# This was made because the user goes Masks > Final Mask > Final GIF, but the selection was not chronological, which meant the wrong Final GIF was being made for a session's regions. 
import os
import numpy as np
from PIL import Image
import imageio.v2 as imageio  # pip install imageio
import pdb

# sessions_to_process = [
#     '2025_10_14_10_25_19_trial_1_TC',
#     '2025_10_14_14_13_15_trial_1_TC',
#     '2025_10_15_10_09_28_trial_1_TC',
#     '2025_10_15_14_16_21_trial_1_TC',
#     '2025_10_16_09_59_15_trial_1_TC',
#     '2025_10_16_14_12_17_trial_1_TC',
#     '2025_10_17_10_05_03_trial_1_TC',
#     '2025_10_17_14_18_23_trial_1_TC',
#     '2025_10_17_14_45_34_trial_1_TP'
# ]

# sessions_to_process = ['2025_10_17_14_45_34_trial_1_TP']
sessions_to_process = ['2025_10_16_14_48_01_trial_1_TP']


MASK_DIR = "/n/holylabs/gershman_lab/Users/zkelso/Masks"
FRAMES_ROOT = "/n/netscratch/gershman_lab/Lab/zkelso/temporary_jpgs/unlabeled-data"
GIF_OUT_DIR = "/n/netscratch/gershman_lab/Lab/zkelso/FINAL_GIF"

fps = 30
mask_color = (0, 255, 0)  # red
mask_alpha = 70           # transparency


def find_final_mask_file(base_sess):
    """
    Find the mask file in MASK_DIR that:
      - contains the base_sess string
      - contains 'FINAL'
      - ends with '.npz'
    Returns full path, or None if not found.
    """
    candidates = [
        f for f in os.listdir(MASK_DIR)
        if base_sess in f and "FINAL" in f and f.endswith(".npz")
    ]
    if not candidates:
        print(f"[{base_sess}] No FINAL mask .npz file found in {MASK_DIR}")
        return None
    if len(candidates) > 1:
        print(f"[{base_sess}] Warning: multiple FINAL mask files found, using most recent:")
        for c in candidates:
            print("   ", c)
            # While this code above suggests the most recent mask would be taken, this was not the case for when I ran the code, hence the addition below.
    
    # OLD CODE: Just took first item from os.listdir() which is arbitrary/random order
    # chosen = candidates[0]
    
    # NEW CODE: Sort by modification time and take the most recent
    candidates.sort(key=lambda f: os.path.getmtime(os.path.join(MASK_DIR, f)), reverse=True)
    chosen = candidates[0] # Default behavior, use unless intentionally choosing another mask
    
    return os.path.join(MASK_DIR, chosen)


def process_one_session(base_sess):
    npz_path = find_final_mask_file(base_sess)
    if npz_path is None:
        return

    # Session+region name comes from the npz filename (without extension)
    sess_full = os.path.splitext(os.path.basename(npz_path))[0]
    sess_full = sess_full.split('_frame')[0]
    print(f"[{base_sess}] Using mask file: {npz_path}")
    print(f"[{base_sess}] Derived inner session folder: {sess_full}")

    frames_dir = os.path.join(FRAMES_ROOT, base_sess, sess_full)
    gif_path = os.path.join(GIF_OUT_DIR, f"{sess_full}_FINAL.gif")

    if not os.path.isdir(frames_dir):
        print(f"[{base_sess}] Frames directory does not exist: {frames_dir}")
        return

    print(f"[{base_sess}] Frames dir: {frames_dir}")
    print(f"[{base_sess}] GIF output: {gif_path}")

    # Load npz
    data = np.load(npz_path)
    masks = data["masks"]  # expected shape: (n_frames, H, W)
    n_frames = masks.shape[0]

    print(f"[{base_sess}] Masks shape: {masks.shape}")

    gif_frames = []

    for i in range(n_frames):
        fname = f"{i:05d}.jpeg"
        fpath = os.path.join(frames_dir, fname)
        if not os.path.exists(fpath):
            print(f"[{base_sess}] Warning: frame not found: {fpath}, skipping")
            continue

        frame = Image.open(fpath).convert("RGB")
        frame_w, frame_h = frame.size

        mask = masks[i]

        # Convert mask to boolean
        if mask.dtype != bool:
            mask_bool = mask > 0
        else:
            mask_bool = mask

        h_mask, w_mask = mask_bool.shape

        # Resize mask to match frame size if needed
        if (w_mask, h_mask) != (frame_w, frame_h):
            mask_img = Image.fromarray(mask_bool.astype(np.uint8) * 255)
            mask_img = mask_img.resize((frame_w, frame_h), resample=Image.NEAREST)
            mask_bool = np.array(mask_img) > 0
            h_mask, w_mask = mask_bool.shape

        # Create overlay
        overlay = Image.new("RGBA", (frame_w, frame_h), (0, 0, 0, 0))
        overlay_pixels = overlay.load()

        for y in range(h_mask):
            for x in range(w_mask):
                if mask_bool[y, x]:
                    overlay_pixels[x, y] = mask_color + (mask_alpha,)

        frame_rgba = frame.convert("RGBA")
        composite = Image.alpha_composite(frame_rgba, overlay)
        composite_rgb = composite.convert("RGB")
        gif_frames.append(np.array(composite_rgb))

        if (i + 1) % 100 == 0:
            print(f"[{base_sess}] Processed {i+1}/{n_frames} frames")

    if gif_frames:
        duration_per_frame = 0.33 / fps  
        os.makedirs(GIF_OUT_DIR, exist_ok=True)
        print(f"Saving gif...")
        imageio.mimsave(gif_path, gif_frames, duration=duration_per_frame)
        print(f"[{base_sess}] Saved GIF to {gif_path}")
    else:
        print(f"[{base_sess}] No frames to save – check your paths and masks.")


def main():
    for sess in sessions_to_process:
        print(f"=== Processing session: {sess} ===")
        process_one_session(sess)


if __name__ == "__main__":
    main()

=== Processing session: 2025_10_16_14_48_01_trial_1_TP ===
[2025_10_16_14_48_01_trial_1_TP] Using mask file: /n/holylabs/gershman_lab/Users/zkelso/Masks/2025_10_16_14_48_01_trial_1_TP_regions_1014_431_1216_1383_fullvideo_frame_split_FINAL_points_binary_masks.npz
[2025_10_16_14_48_01_trial_1_TP] Derived inner session folder: 2025_10_16_14_48_01_trial_1_TP_regions_1014_431_1216_1383_fullvideo
[2025_10_16_14_48_01_trial_1_TP] Frames dir: /n/netscratch/gershman_lab/Lab/zkelso/temporary_jpgs/unlabeled-data/2025_10_16_14_48_01_trial_1_TP/2025_10_16_14_48_01_trial_1_TP_regions_1014_431_1216_1383_fullvideo
[2025_10_16_14_48_01_trial_1_TP] GIF output: /n/netscratch/gershman_lab/Lab/zkelso/FINAL_GIF/2025_10_16_14_48_01_trial_1_TP_regions_1014_431_1216_1383_fullvideo_FINAL.gif
[2025_10_16_14_48_01_trial_1_TP] Masks shape: (1799, 952, 202)
[2025_10_16_14_48_01_trial_1_TP] Processed 100/1799 frames
[2025_10_16_14_48_01_trial_1_TP] Processed 200/1799 frames
[2025_10_16_14_48_01_trial_1_TP] Processed